In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

# 06 — Class Imbalance Handling
Compare SMOTE, Random Undersampling, and Class Weighting across regularization types. Focus on precision-recall tradeoff.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from src.data_loader import get_all_datasets
from src.preprocessing import get_pipelines
from src.imbalance import (
    build_smote_pipeline, build_undersample_pipeline,
    build_class_weight_pipeline, cross_val_imbalance
)
from src.comparative_analysis import build_imbalance_interaction_table
from src.evaluation import save_table
from src.utils import RANDOM_STATE

datasets = get_all_datasets()
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
best_C = 1.0

holdout = {}
for name, (X, y) in datasets.items():
    X_main, X_test, y_main, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    holdout[name] = (X_main, y_main, X_test, y_test)

2026-05-15 22:18:29,736 — Loading Bonn EEG (extracted 6-dim features)...


In [3]:
# Run imbalance experiments
imb_results = {}

for dataset_name, (X, y, _, _) in holdout.items():
    if dataset_name == 'bonn_eeg':
        continue  # Bonn is balanced
    pipe = get_pipelines(dataset_name)['A']

    for imb_method in ['SMOTE', 'Undersample', 'ClassWeight']:
        for reg_type in ['L1', 'L2', 'ElasticNet']:
            model_kwargs = {
                'penalty': 'l1' if reg_type == 'L1' else ('l2' if reg_type == 'L2' else 'elasticnet'),
                'solver': 'saga' if reg_type in ['L1', 'ElasticNet'] else 'lbfgs',
                'C': best_C, 'max_iter': 5000, 'random_state': RANDOM_STATE,
            }
            if reg_type == 'ElasticNet':
                model_kwargs['l1_ratio'] = 0.5

            if imb_method == 'SMOTE':
                model = LogisticRegression(**model_kwargs)
                imb_pipe = build_smote_pipeline(pipe, model)
            elif imb_method == 'Undersample':
                model = LogisticRegression(**model_kwargs)
                imb_pipe = build_undersample_pipeline(pipe, model)
            else:
                imb_pipe = build_class_weight_pipeline(pipe, model_kwargs)

            results = cross_val_imbalance(imb_pipe, X, y, CV)
            imb_results[(imb_method, reg_type, dataset_name)] = results
print(f'Imbalance experiments complete: {len(imb_results)} entries')

c:\Users\deepthink\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\deepthink\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\deepthink\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead

KeyboardInterrupt: 

In [ ]:
# Table 4: Imbalance × Regularization interaction
table4 = build_imbalance_interaction_table(imb_results)
save_table(table4, 'table4_imbalance_interaction')
print(table4.to_string(index=False))

In [ ]:
# Grouped bar chart: PR-AUC by imbalance method × regularization (UCI Seizure)
methods = ['SMOTE', 'Undersample', 'ClassWeight']
reg_types = ['L1', 'L2', 'ElasticNet']
x = np.arange(len(reg_types))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, method in enumerate(methods):
    means = [imb_results.get((method, rt, 'uci_seizure'), {}).get('pr_auc', np.array([0])).mean()
             for rt in reg_types]
    stds = [imb_results.get((method, rt, 'uci_seizure'), {}).get('pr_auc', np.array([0])).std()
            for rt in reg_types]
    ax.bar(x + i * width, means, width, yerr=stds, label=method, capsize=4)

ax.set_xticks(x + width)
ax.set_xticklabels(reg_types)
ax.set_ylabel('PR-AUC (mean ± std)')
ax.set_title('Imbalance Method × Regularization — UCI Seizure (Pipeline A, C=1)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../results/figures/imbalance_interaction.png', dpi=150)
plt.show()

In [ ]:
# Precision vs Recall tradeoff per imbalance method (L2, UCI Seizure)
fig, ax = plt.subplots(figsize=(8, 5))
for method, color in zip(methods, ['steelblue', 'tomato', 'green']):
    key = (method, 'L2', 'uci_seizure')
    if key in imb_results:
        r = imb_results[key]
        ax.scatter(r['recall'].mean(), r['pr_auc'].mean(), s=120, color=color, label=method, zorder=5)
        ax.errorbar(r['recall'].mean(), r['pr_auc'].mean(),
                    xerr=r['recall'].std(), yerr=r['pr_auc'].std(),
                    color=color, capsize=4, alpha=0.6)

ax.set_xlabel('Recall (mean ± std)')
ax.set_ylabel('PR-AUC (mean ± std)')
ax.set_title('Precision-Recall Tradeoff by Imbalance Method (L2, UCI Seizure)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/pr_tradeoff_imbalance.png', dpi=150)
plt.show()